# 🚀 SGLang 部署与调优实战

**本文目标**：掌握 SGLang 的生产部署、性能调优和与 vLLM 的选型决策。

## 1. 部署

```bash
# 安装
pip install sglang[all]

# 启动 (单 GPU)
python -m sglang.launch_server \
    --model-path meta-llama/Llama-3-8B-Instruct \
    --host 0.0.0.0 --port 30000

# 多 GPU (tensor parallel)
python -m sglang.launch_server \
    --model-path Llama-3-70B-Instruct \
    --tp-size 4

# Docker
docker run --gpus all -p 30000:30000 \
    lmsysorg/sglang:latest \
    python -m sglang.launch_server \
    --model-path /models/model-name \
    --host 0.0.0.0 --port 30000
```

## 2. 核心参数

| 参数 | 默认 | 作用 | Agent 场景建议 |
|------|------|------|---------------|
| --context-length | 自动 | 最大 context | 16384+ |
| --tp-size | 1 | Tensor parallel | 70B 模型用 4 |
| --mem-fraction-static | 0.88 | 显存使用率 | 0.85 |
| --max-running-requests | 可配 | 最大并发 | 16-32 |
| --schedule-policy | lpm | 调度策略 | lpm (默认, 就适合 Agent) |
| --enable-radix-cache | True | RadixAttention | 保持开启 |
| --chunked-prefill-size | 可配 | Prefill chunk | 4096 |

## 3. 调度策略

```bash
# SGLang 的三种调度策略
--schedule-policy lpm      # Longest Prefix Match (默认, 推荐)
--schedule-policy priority  # 按优先级
--schedule-policy fcfs      # 先到先服务

# LPM 的工作原理:
# 在 waiting 队列中, 优先调度与当前 running 请求有最长公共前缀的请求
# → 最大化 Radix Tree 的命中率
# → Agent 场景收益最大
```

## 4. 监控

```bash
# SGLang 的 metrics endpoint
curl http://localhost:30000/metrics

# 关键指标:
sglang:num_running_reqs
sglang:num_queue_reqs
sglang:radix_cache_hit_rate       # ★ 核心指标
sglang:radix_cache_size_bytes
sglang:time_to_first_token_seconds
sglang:time_per_output_token_seconds
sglang:gen_throughput
```

## 5. SGLang vs vLLM 选型决策

| 场景 | 推荐 | 理由 |
|------|------|------|
| 通用文本 API | vLLM | 生态更完善, 社区更大 |
| Agent 密集型 | **SGLang** | RadixAttention + constrained decoding |
| 批量提取/分类 | **SGLang** | 约束生成 + 前缀共享 |
| 多模态 (同图复用多) | **SGLang** | visual token 自动共享 |
| 多模态 (图多样) | vLLM | 生态支持更广 |
| 多模型/多 LoRA | vLLM | LoRA 支持更成熟 |
| 消费级 GPU | SGLang (性能稍好) | 但两者都可用 |
| 固定模型/极致吞吐 | TensorRT-LLM | 编译优化 |

### 总结

```
SGLang 的核心优势在三个维度:
  1. Token-level prefix sharing (Radix Tree)
  2. First-class constrained generation (FSM)
  3. 组合优势: constrained gen → 一致的 token 序列 → 更高的 cache 命中

vLLM 的优势:
  1. 更大的社区和生态
  2. 更完善的 LoRA/多模型支持
  3. 更成熟的生产部署方案

建议:
  - 如果是 Agent 应用 → SGLang (没有争议)
  - 如果是通用 API → vLLM (默认选择)
  - 两者都部署 → 按请求类型路由
```